# Polygon Trade Analysis

Loads raw per-side trades from `data/trades_polygon/*.parquet` and markets from
`data/markets/*.parquet`, then performs wallet-level P&L analysis and extracts trades
of the **top 5% wallets** identified on the training period.

### Data schema notes

Trades are stored as **wallet-perspective rows** (already split per side), with key fields:
`wallet`, `side`, `token_id`, `quantity`, `price`, `usdc_amount`.

So each parquet row is already one wallet-side fill (no additional expansion step needed).

Final P&L per wallet is computed from per-trade P&L:
- `BUY`: `final_value_usdc - trade_value_usdc`
- `SELL`: `trade_value_usdc - final_value_usdc`

where `final_value_usdc = quantity × final_price`, and `final_price` is 1.0 for winning
outcome tokens, 0.0 otherwise.


In [60]:
%load_ext autoreload
%autoreload 2

import json
import datetime
from pathlib import Path

import pandas as pd

import numpy as np
np.set_printoptions(precision=4, suppress=True)
pd.options.display.float_format = '{:.2f}'.format

# try to use orjson for faster JSON parsing (install with: pip install orjson)
try:
    import orjson
    def json_loads(s):
        return orjson.loads(s)
except ImportError:
    def json_loads(s):
        return json.loads(s)

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Parameters

In [61]:
# ── market end_date filter (inclusive) ───────────────────────────────────────
# Only markets whose end_date falls within [START_DATE, END_DATE] are loaded.
# Trades on tokens from markets outside this range are excluded.
START_DATE = datetime.date(2025, 1, 1)
END_DATE   = datetime.date(2027, 4, 5)

# ── train / test split ────────────────────────────────────────────────────────
# Trades with dt.date <= END_DATE_TRAIN are flagged is_train=True.
# Top-4%-per-shard wallet selection is performed on training data only.
END_DATE_TRAIN = datetime.date(2026, 2, 15)

COPY_WINDOW_SECONDS = 5 * 60  # 5 minutes

# ── input paths ───────────────────────────────────────────────────────────────
TRADES_DIR  = Path("../../data/trades_polygon")
ENRICHED_TRADES_DIR  = Path("../../data/trades_polygon_enriched")
MARKETS_DIR = Path("../../data/markets")

# ── output directory (partitioned parquet shards) ────────────────────────────
OUT_DIR = Path("../../data/polygon_trades_processed")


## 1 · Load markets

Each `data/markets/YYYY-MM.parquet` file has three columns:
- `end_date_iso` – market resolution date string
- `condition_id` – unique market identifier
- `market_json` – full market definition as a JSON string

The token lookup table (`token_id → condition_id, outcome, token_winner, final_price`) is
built once via `build_token_lookup_df()` and persisted to
`data/markets_processed/token_lookup.parquet`. Subsequent runs load it with
`load_token_lookup_df()` instead of re-parsing all market JSON.

In [65]:
# # Load only the market files whose month falls within [START_DATE, END_DATE].
# import datetime as _dt
# def _file_in_range(p, start, end):
#     """Return True if YYYY-MM.parquet filename falls within the date range."""
#     try:
#         y, m = (int(x) for x in p.stem.split("-"))
#         file_date = _dt.date(y, m, 1)
#         return start.replace(day=1) <= file_date <= end.replace(day=1)
#     except Exception:
#         return False

# market_files = [
#     p for p in sorted(MARKETS_DIR.glob("*.parquet"))
#     if _file_in_range(p, START_DATE, END_DATE)
# ]
# print(f"Found {len(market_files)} market file(s)")

# all_market_rows = pd.concat(
#     [pd.read_parquet(f) for f in market_files],
#     ignore_index=True,
# )
# # deduplicate by condition_id (keep first occurrence)
# all_market_rows = all_market_rows.drop_duplicates(subset="condition_id", keep="first")
# print(f"Unique markets (all):  {len(all_market_rows):,}")

# # ── apply START_DATE / END_DATE filter ───────────────────────────────────────
# # Parse end_date_iso as a date; keep only markets whose resolution date falls
# # within [START_DATE, END_DATE] (inclusive).
# end_dates = pd.to_datetime(all_market_rows["end_date_iso"], utc=True, errors="coerce").dt.date
# in_range  = (end_dates >= START_DATE) & (end_dates <= END_DATE)
# all_market_rows = all_market_rows[in_range].reset_index(drop=True)
# print(f"Unique markets (filtered {START_DATE} → {END_DATE}): {len(all_market_rows):,}")
# all_market_rows.head(2)

In [66]:
from polymarket_analysis.data.data_catalogue import (
    load_markets_processed,
    build_token_lookup_df,
    load_token_lookup_df,
)
token_df = load_token_lookup_df()
print(f"Token lookup rows: {len(token_df):,}")

token_df = token_df[
    ~(token_df['primary_tag'].isin(['Sports', 'Crypto']))
    & token_df['token_winner'].notna()
    ]
len(token_df)

Token lookup rows: 1,327,030


200496

## 2 · Process trades — Phase 1: select top-4% wallets per shard

Each shard is processed independently in parallel.  Only per-wallet training-period P&L
is returned (no trade rows), so memory usage is minimal.  The union of top-4% wallets
across all shards becomes the candidate set for Phase 2.

In [67]:
# Preprocess: enrich by copyable quantity if not present

from concurrent.futures import ProcessPoolExecutor, as_completed

from polymarket_analysis.preprocessing.fill_extender import enrich_shard

trade_files = sorted(TRADES_DIR.glob("*.parquet"))

with ProcessPoolExecutor() as executor:
    futures = {executor.submit(enrich_shard, f, ENRICHED_TRADES_DIR, COPY_WINDOW_SECONDS, token_df): f for f in trade_files}

    total = len(futures)

    for i, fut in enumerate(as_completed(futures), start=1):
        f = futures[fut]
        try:
            fut.result()
        except Exception as e:
            print(f"Error processing {f}: {e}")
            continue

        if i % 4 == 0 or i == total:
            print(f"{i}/{total} shards processed")

7818971 trades in 0.parquet
7915325 trades in 5.parquet
8487586 trades in 7.parquet
8603313 trades in 6.parquet
8013490 trades in 8.parquet
8098868 trades in 9.parquet
1723714 trades after merging with token_df for 5.parquet
1964837 trades after merging with token_df for 0.parquet
2125982 trades after merging with token_df for 7.parquet
1722396 trades after merging with token_df for 9.parquet
2170345 trades after merging with token_df for 8.parquet
2524317 trades after merging with token_df for 6.parquet
8109445 trades in 4.parquet
7947854 trades in 1.parquet
2254618 trades after merging with token_df for 4.parquet
8286071 trades in 3.parquet
2155234 trades after merging with token_df for 3.parquet
1883662 trades after merging with token_df for 1.parquet
8449069 trades in 2.parquet
2366885 trades after merging with token_df for 2.parquet
Enriched 5.parquet with copyable_qty
Enriched 9.parquet with copyable_qty
8335726 trades in a.parquet
7885352 trades in b.parquet
2480854 trades after

In [68]:
# pd.set_option('display.max_colwidth', None)
# pd.set_option('display.max_columns', None)
# pd.set_option('display.max_rows', 200)

# MARKET = '0x02d03a859b9d64c7cbbc2a2c3172898bb89b379dba10d54d74ee2c1d5036a71b'

# df = pd.read_parquet(ENRICHED_TRADES_DIR / "enriched_0.parquet")
# df = df[df['condition_id'] == MARKET]
# df[['side', 'price', 'quantity', 'ts', 'token_id', 'tx_hash', 'wallet', 'position']].head(5)

In [69]:
# len(df)
# df[df['tx_hash']=='0xffece5c5cde2b0e1b9375cf30cbb3af09f087967143aff3b4fe5e53c8d1b58c3']

In [70]:
# df['ts'] = pd.to_datetime(df['block_timestamp'], unit='s')
# df['token_id'] = df['token_id'].str[:5]
# df['tx_hash'] = df['tx_hash'].str[:5]
# df[["wallet", 'tx_hash', 'ts', "side", "price", "quantity","token_id", "token_winner", "avail_copy_qty", "avail_copy_total_vol", "avail_copy_count","condition_id"]].head(1)

In [71]:
# ── load token resolution DataFrame ─────────────────────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)

enriched_trade_files = sorted(ENRICHED_TRADES_DIR.glob("*.parquet"))

# ── Phase 1: identify top-4% wallets per shard (parallel) ────────────────────
print("Phase 1 — selecting top-4% wallets per shard...")
shard_wallet_pnls: dict[str, float] = {}   # wallet → best shard pnl (for diagnostics)
shard_wallet_thresholds: dict[int, float] = {}   # shard index → top-pct threshold (for diagnostics)
total_raw_rows      = 0
total_in_range_rows = 0
total_candidates    = 0

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(select_top_wallets_shard, f, token_df, END_TRAIN_TS, top_pct=0.04): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        wallet_pnl, stats = fut.result()
        total_raw_rows      += stats["raw_rows"]
        total_in_range_rows += stats["in_range_rows"]
        total_candidates    += stats["candidate_wallets"]
        # union: keep the wallet; if it appears in multiple shards take max pnl
        for w, pnl in wallet_pnl.items():
            if w not in shard_wallet_pnls or pnl > shard_wallet_pnls[w]:
                shard_wallet_pnls[w] = pnl
        shard_wallet_thresholds[i] = stats["threshold"]
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(
                f"  {i}/{len(enriched_trade_files)} shards | "
                f"raw: {total_raw_rows:,} | in-range: {total_in_range_rows:,} | "
                f"wallet union so far: {len(shard_wallet_pnls):,}"
            )

top_wallets: set[str] = set(shard_wallet_pnls.keys())
print(f"\nPhase 1 done. Candidate wallets (union of top-4% per shard): {len(top_wallets):,}")
print(f"Threshold pnls per shard: {shard_wallet_thresholds}")

Phase 1 — selecting top-4% wallets per shard...
Processing shard enriched_0.parquet...
Processing shard enriched_1.parquet...
Shard top 4.00% threshold: 22.47 USDC
Processing shard enriched_2.parquet...
Shard top 4.00% threshold: 18.36 USDC
Processing shard enriched_3.parquet...
Shard top 4.00% threshold: 30.81 USDC
Processing shard enriched_4.parquet...
Shard top 4.00% threshold: 36.14 USDC
Processing shard enriched_5.parquet...
  4/16 shards | raw: 8,370,618 | in-range: 8,370,618 | wallet union so far: 19,643
Processing shard enriched_6.parquet...
Shard top 4.00% threshold: 37.23 USDC
Shard top 4.00% threshold: 13.80 USDC
Processing shard enriched_7.parquet...
Processing shard enriched_8.parquet...
Shard top 4.00% threshold: 43.14 USDC
Shard top 4.00% threshold: 21.90 USDC
Processing shard enriched_9.parquet...
  8/16 shards | raw: 16,999,249 | in-range: 16,999,249 | wallet union so far: 32,959
Shard top 4.00% threshold: 33.32 USDC
Processing shard enriched_a.parquet...
Shard top 4.0

## 3 · Phase 2: enrich, group by tx×wallet×side, filter to top wallets

Each shard is processed in parallel.  Fills are inner-joined with settlement data,
aggregated to one row per ``tx_hash × wallet × side``, and filtered to the wallet set
from Phase 1.  Results are concatenated in memory and re-grouped to merge any
transactions that span shard boundaries.

In [72]:
from polymarket_analysis.preprocessing.shard_processor import enrich_and_group_shard

# ── Phase 2: enrich + group + filter (parallel) ──────────────────────────────
print("Phase 2 — grouping and filtering shards...")
shard_results:     list[pd.DataFrame]    = []
wallet_pnl_total:  dict[str, float]      = {}   # accumulated training P&L per wallet
sample_fills = None

with ProcessPoolExecutor() as executor:
    futures = {
        executor.submit(enrich_and_group_shard, f, token_df, END_TRAIN_TS, top_wallets): f
        for f in enriched_trade_files
    }
    for i, fut in enumerate(as_completed(futures), start=1):
        grouped_shard, shard_train_pnl = fut.result()
        if not grouped_shard.empty:
            shard_results.append(grouped_shard)
            if sample_fills is None:
                sample_fills = grouped_shard.head(5).copy()
        for w, pnl in shard_train_pnl.items():
            wallet_pnl_total[w] = wallet_pnl_total.get(w, 0.0) + pnl
        if i % 4 == 0 or i == len(enriched_trade_files):
            print(f"  {i}/{len(enriched_trade_files)} shards | shards with data: {len(shard_results)}")

if not shard_results:
    raise ValueError("No in-range trade rows after token filter")

# ── merge cross-shard partial groups ─────────────────────────────────────────
GROUP_KEYS = ["tx_hash", "wallet", "side"]
grouped = pd.concat(shard_results, ignore_index=True)
grouped = (
    grouped.groupby(GROUP_KEYS, sort=False)
    .agg(
        dt               = ("dt",               "first"),
        condition_id     = ("condition_id",     "first"),
        outcome          = ("outcome",          "first"),
        token_winner     = ("token_winner",     "first"),
        final_price      = ("final_price",      "first"),
        position         = ("position",         "max"),
        total_quantity   = ("total_quantity",   "sum"),
        price_x_qty_sum  = ("price_x_qty_sum",  "sum"),
        trade_value_usdc = ("trade_value_usdc", "sum"),
        final_value_usdc = ("final_value_usdc", "sum"),
        num_fills        = ("num_fills",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
        copyable_pnl     = ("copyable_pnl",     "sum"),
        copyable_qty    = ("copyable_qty",   "sum"),
        avail_copy_total_vol = ("avail_copy_total_vol", "sum"),
        avail_copy_count  = ("avail_copy_count", "sum"),
    )
    .reset_index()
)
grouped["avg_price"] = grouped["price_x_qty_sum"] / grouped["total_quantity"]
# grouped.drop(columns=["price_x_qty_sum"], inplace=True)
grouped.sort_values(["wallet", "dt"], inplace=True, ignore_index=True)

print(f"\nPhase 2 done.")
print(f"  Grouped rows (all top wallets, all periods): {len(grouped):,}")
print(f"  Unique wallets in grouped:                   {grouped['wallet'].nunique():,}")
grouped.head(5)

Phase 2 — grouping and filtering shards...
  4/16 shards | shards with data: 4
  8/16 shards | shards with data: 8
  12/16 shards | shards with data: 12
  16/16 shards | shards with data: 16

Phase 2 done.
  Grouped rows (all top wallets, all periods): 16,524,106
  Unique wallets in grouped:                   51,444


,tx_hash,wallet,side,dt,condition_id,outcome,token_winner,final_price,position,total_quantity,price_x_qty_sum,trade_value_usdc,final_value_usdc,num_fills,trade_pnl,copyable_pnl,copyable_qty,avail_copy_total_vol,avail_copy_count,avg_price
0,0x9fb5ad77b60ada07898f68d1d1bd959f924789b918421cd3284ae41599881a3c,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 06:27:41+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,10435.00,10435.00,20.87,20.87,0.00,2,-20.87,0.00,-0.00,-0.00,0.00,0.00
1,0xf9c6f7aa5883f574563ec4ba53fa49f5b1fcd43cb86a09fa06b496a2c5672e4e,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 07:25:19+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,20870.00,10435.00,20.87,20.87,0.00,2,-20.87,-0.00,0.00,-0.00,0.00,0.00
2,0x31d4022f503889f8acdbbb0f93b4db52fa80b454deca1026bc0dadb4af48aa80,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:03+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,18007.01,2862.99,5.73,5.73,0.00,1,5.73,5.73,2862.99,8.65,2.00,0.00
3,0x04705dba811ad7cbc99d7482b2f9b71319117c66ea7535d7b6b33d1869e919f6,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:53+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,10435.00,7572.01,15.14,15.14,0.00,1,15.14,15.14,7572.01,28.69,3.00,0.00
4,0xfb8fd88523629a26116f05c1f72c7a36e6aa386640d29cf5f84d87171df212c9,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 09:11:33+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,False,0.00,20650.00,10215.00,20.43,20.43,0.00,2,-20.43,-0.00,0.00,-0.00,0.00,0.00


## 4 · Phase 3: apply final PnL cut-off and export

Compute the true cross-shard training P&L per wallet.  Use the **median** of the
per-shard P&L values accumulated in Phase 1 as the export cut-off: wallets whose
total training P&L falls below that median are dropped before writing.

In [73]:
# ── wallet summary (full cross-shard training P&L) ───────────────────────────
END_TRAIN_TS = pd.Timestamp(END_DATE_TRAIN, tz="UTC") + pd.Timedelta(days=1)
grouped_train = grouped[grouped["dt"] < END_TRAIN_TS]

wallet_summary = (
    grouped_train.groupby("wallet")
    .agg(
        num_markets     = ("condition_id",    "nunique"),
        num_trades      = ("num_fills",        "sum"),
        total_cost_usdc = ("trade_value_usdc", "sum"),
        copyable_pnl       = ("copyable_pnl",        "sum"),
        trade_pnl        = ("trade_pnl",        "sum"),
    )
    .sort_values("copyable_pnl", ascending=False)
    .reset_index()
)

# ── PnL cut-off: median of the per-shard pnl values collected in Phase 1 ─────
import statistics
pnl_cutoff = statistics.median(shard_wallet_thresholds.values())

print(f"\nUnique wallets in training data: {wallet_summary['wallet'].nunique():,}")
print(f"wallet_summary['copyable_pnl'] quantiles:\n{wallet_summary['copyable_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}")

final_wallets = set(
    wallet_summary[
        (wallet_summary["copyable_pnl"] >= pnl_cutoff) &
        (wallet_summary["trade_pnl"] >= wallet_summary["copyable_pnl"])
    ]["wallet"]
)

print(f"\nFinal selected wallets (Phase 2 filter): {len(final_wallets):,}")
print(f"final_wallets['copyable_pnl'] quantiles:\n{wallet_summary[wallet_summary['wallet'].isin(final_wallets)]['copyable_pnl'].quantile([0.01, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.99])}")

print(f"Unique wallets after Phase 2 union:    {len(wallet_summary):,}")
print(f"Per-shard PnL median (cut-off):        {pnl_cutoff:,.2f} USDC")
print(f"Wallets passing PnL cut-off:           {len(final_wallets):,}")
wallet_summary.head(5)


Unique wallets in training data: 51,444
wallet_summary['copyable_pnl'] quantiles:
0.01   -6822.36
0.10    -450.71
0.20     -83.84
0.30       4.11
0.40      30.40
0.50      48.05
0.60      73.03
0.70     118.05
0.80     225.47
0.90     587.54
0.99    7556.30
Name: copyable_pnl, dtype: float64

Final selected wallets (Phase 2 filter): 15,720
final_wallets['copyable_pnl'] quantiles:
0.01      23.79
0.10      35.12
0.20      46.81
0.30      61.11
0.40      82.23
0.50     115.18
0.60     169.10
0.70     276.71
0.80     519.52
0.90    1307.03
0.99   16794.61
Name: copyable_pnl, dtype: float64
Unique wallets after Phase 2 union:    51,444
Per-shard PnL median (cut-off):        22.61 USDC
Wallets passing PnL cut-off:           15,720


,wallet,num_markets,num_trades,total_cost_usdc,copyable_pnl,trade_pnl
0,0x17db3fcd93ba12d38382a0cade24b200185c5f6d,42,2385,3878059.88,501531.92,1304494.87
1,0x2a21fec355312b609b9ac97ce4eda3e93ec3d307,17,1069,2372304.03,240110.13,268316.85
2,0x1f1d38f54a615d1c43c77e3b109bd01f56e163d9,17,2547,1348382.69,222966.54,688774.49
3,0x551e72eda42a5ab39d6d78239a1d9bbb5db6b0e0,313,5926,3521854.08,154224.66,282061.18
4,0xbacd00c9080a82ded56f504ee8810af732b0ab35,1059,109611,23214183.72,140370.19,645638.20


## 5 · Market-level volume summary

In [74]:
# join market metadata (question, end_date) for display
markets_meta = load_markets_processed()[["condition_id", "question", "end_date_iso"]].rename(
    columns={"end_date_iso": "end_date"}
)
grouped_meta = grouped.merge(
    markets_meta,
    on="condition_id",
    how="left",
)

market_summary = (
    grouped_meta.groupby(["condition_id", "question", "end_date"])
    .agg(
        num_wallets      = ("wallet",           "nunique"),
        num_trades       = ("num_fills",         "sum"),
        volume_usdc      = ("trade_value_usdc",  "sum"),
        final_value_usdc = ("final_value_usdc",  "sum"),
    )
    .reset_index()
    .sort_values("volume_usdc", ascending=False)
    .reset_index(drop=True)
)

print(f"Unique markets: {len(market_summary):,}")
market_summary.head(10)

Unique markets: 29,003


,condition_id,question,end_date,num_wallets,num_trades,volume_usdc,final_value_usdc
0,0x43ec78527bd98a0588dd9455685b2cc82f5743140cb3a154603dc03c02b57de5,US government shutdown Saturday?,2026-01-31T00:00:00Z,6451,261767,130640342.21,134337035.82
1,0x655e5ca101c466b6293aa15e06173b78b293221803d56e35551f708cd82eb352,Will Zelenskyy wear a suit before July?,2025-06-30T00:00:00Z,2513,76708,128916231.77,129337568.31
2,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Khamenei out as Supreme Leader of Iran by February 28?,2026-02-28T00:00:00Z,5019,170604,93400354.00,101610465.09
3,0xef8cf8b45ef7e3a607a72b6e1d56bede869fdd81795b63db847de1090bf11c41,TikTok banned in the US before May 2025?,2025-04-30T00:00:00Z,1721,42800,73139738.21,73882661.22
4,0x3488f31e6449f9803f99a8b5dd232c7ad883637f1c86e6953305a2ef19c77f20,"US strikes Iran by February 28, 2026?",2026-01-31T00:00:00Z,5997,187832,63056248.75,69611505.59
5,0x32707ae194389a70f25314ec934851daba12d893da616c492ffe8b655b7759a9,"Will Eleven die in ""Stranger Things: Season 5""?",2025-12-31T00:00:00Z,1763,38075,61068250.70,61080798.12
6,0x77b4a1d4cd398a86c18b6bb9b5218917dc9f04b01a3cd4bae1c71ea345f15fa8,Will Polymarket US go live in 2025?,2025-12-31T00:00:00Z,3644,101098,47414361.58,47960371.06
7,0x62b0cd598091a179147acbd4616400f804acfdff6f76f029944b481b37cbd45f,US x Venezuela military engagement by December 31?,2025-12-31T00:00:00Z,3799,118040,45453623.13,44736595.65
8,0x8ee2f1640386310eb5e7ffa596ba9335f2d324e303d21b0dfea6998874445791,Russia x Ukraine ceasefire in 2025?,2025-12-31T00:00:00Z,5729,148701,45260579.52,43734013.30
9,0x61ce3773237a948584e422de72265f937034af418a8b703e3a860ea62e59ff36,Will the Iranian regime fall by March 31?,2026-03-31T00:00:00Z,3679,186972,44437847.08,45161022.96


## 6 · Wallet statistics (quantiles)

In [75]:
QUANTILES = [0.001, 0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99, 0.999, 1]
N = len(wallet_summary)

wallet_stats = (
    wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]]
    .quantile(QUANTILES)
    .rename_axis("quantile")
)

wallet_stats.insert(0, "wallet_count", [round(q * N) for q in QUANTILES])
wallet_stats.loc["count"] = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].count()
wallet_stats.loc["mean"]  = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].mean()
wallet_stats.loc["std"]   = wallet_summary[["num_trades", "num_markets", "copyable_pnl", "trade_pnl"]].std()
wallet_stats.loc["count", "wallet_count"] = N
wallet_stats.loc["mean",  "wallet_count"] = float("nan")
wallet_stats.loc["std",   "wallet_count"] = float("nan")
wallet_stats["wallet_count"] = wallet_stats["wallet_count"].astype("Int64")
wallet_stats["wallet_count_complement"] = N - wallet_stats["wallet_count"]
wallet_stats["num_trades"]  = wallet_stats["num_trades"].round(1)
wallet_stats["num_markets"] = wallet_stats["num_markets"].round(1)
wallet_stats["copyable_pnl"]    = wallet_stats["copyable_pnl"].round(2)
wallet_stats["trade_pnl"]    = wallet_stats["trade_pnl"].round(2)
wallet_stats

,wallet_count,num_trades,num_markets,copyable_pnl,trade_pnl,wallet_count_complement
quantile,,,,,,
0.00,51,1.00,1.00,-38798.13,-80700.03,51393
0.01,514,1.00,1.00,-6822.36,-14288.89,50930
0.05,2572,3.00,1.00,-1237.97,-2440.04,48872
0.10,5144,5.00,1.00,-450.71,-912.05,46300
0.25,12861,13.00,3.00,-28.09,-134.82,38583
0.50,25722,41.00,8.00,48.05,29.79,25722
0.75,38583,134.00,22.00,157.61,227.90,12861
0.90,46300,431.00,59.00,587.54,1244.19,5144
0.95,48872,971.00,111.00,1323.28,3587.99,2572


## 7 · Full enriched trade table

In [76]:
DISPLAY_COLS = [
    "tx_hash", "wallet", "side",
    "dt", "condition_id", "outcome", "position", "total_quantity", "avg_price",
    "token_winner", "final_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl", "num_fills",
]

grouped[DISPLAY_COLS]

,tx_hash,wallet,side,dt,condition_id,outcome,position,total_quantity,avg_price,token_winner,final_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,num_fills
0,0x9fb5ad77b60ada07898f68d1d1bd959f924789b918421cd3284ae41599881a3c,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 06:27:41+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,10435.00,10435.00,0.00,False,0.00,20.87,0.00,-20.87,0.00,2
1,0xf9c6f7aa5883f574563ec4ba53fa49f5b1fcd43cb86a09fa06b496a2c5672e4e,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 07:25:19+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,20870.00,10435.00,0.00,False,0.00,20.87,0.00,-20.87,-0.00,2
2,0x31d4022f503889f8acdbbb0f93b4db52fa80b454deca1026bc0dadb4af48aa80,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:03+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,18007.01,2862.99,0.00,False,0.00,5.73,0.00,5.73,5.73,1
3,0x04705dba811ad7cbc99d7482b2f9b71319117c66ea7535d7b6b33d1869e919f6,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,SELL,2025-11-04 07:26:53+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,10435.00,7572.01,0.00,False,0.00,15.14,0.00,15.14,15.14,1
4,0xfb8fd88523629a26116f05c1f72c7a36e6aa386640d29cf5f84d87171df212c9,0x00000000d1e5cf1790f4ab99b1e2b1cf87ad9574,BUY,2025-11-04 09:11:33+00:00,0x16d0091ceb7205446bbad250bf51f4f0d9f95265f8945422289c97dc4926cffd,No,20650.00,10215.00,0.00,False,0.00,20.43,0.00,-20.43,-0.00,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
16524101,0x3bcd973a3ee5617ac8b063536a42ebc524c9863ceaab5833c57501df536833ca,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-08 18:51:25+00:00,0xc734f61dbb7fbf5ff7afac8c18b1f05f13c4b9b408f8f8d6d210a862ebabd00d,Yes,2500.00,1000.00,0.01,False,0.00,6.00,0.00,-6.00,-6.00,1
16524102,0x070c8061d33e893ea6c921e020c1fc3bd318e437687c5e19be1c3ea3b211eddd,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 20:24:18+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,200.00,200.00,0.02,False,0.00,4.00,0.00,-4.00,-4.00,1
16524103,0x852015bc27fa54364ba4bd307b710982f8c475e33ff5e2ff1c46933b30c74547,0xffffffe1e093aacd21e4e281e66d543fb0b23455,BUY,2026-02-18 23:00:20+00:00,0x4f7faa55b26773289b8253c2cb587a8bf880f083c79963076e524c599d03efda,Yes,4200.00,4000.00,0.00,False,0.00,12.00,0.00,-12.00,-12.00,1
16524104,0x8815e825e712da69beadc37cf8ddc48f2eb2ce9eaec7c859f3fc1bd1ce740b6f,0xffffffe1e093aacd21e4e281e66d543fb0b23455,SELL,2026-02-28 16:39:01+00:00,0xd4bbf7f6707c67beb736135ad32a41f6db41f8ae52d3ac4919650de9eeb94ed8,Yes,530.00,50.00,0.51,True,1.00,25.35,50.00,-24.65,-24.65,2


## 8 · Export: apply PnL cut-off and write partitioned parquet

Filter grouped trades to ``final_wallets`` (wallets above the median per-shard PnL),
then write one parquet shard per first hex character of ``condition_id`` after ``0x``.

In [77]:
top_wallets = final_wallets
print(f"Wallets selected for export: {len(top_wallets):,}")

Wallets selected for export: 15,720


In [78]:
import shutil

EXPORT_COLS = [
    "wallet", "condition_id", "dt",
    "side", "outcome", "position",
    "total_quantity", "avg_price",
    "trade_value_usdc", "final_value_usdc", "trade_pnl", "copyable_pnl",
    "token_winner", "final_price",
    "tx_hash", "num_fills",
    "is_train","copyable_qty", "avail_copy_total_vol", "avail_copy_count",
]

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
OUT_DIR.mkdir(parents=True, exist_ok=True)

export_df = grouped[grouped["wallet"].isin(top_wallets)].copy()
export_df["is_train"] = export_df["dt"].dt.date <= END_DATE_TRAIN
export_df = export_df[EXPORT_COLS].reset_index(drop=True)

# Partition by the first hex character of condition_id after the "0x" prefix,
# matching the input shard layout (0.parquet … 9.parquet, a.parquet … f.parquet).
export_df["_shard"] = export_df["condition_id"].str[2]

output_files = []
for shard_key, shard_df in export_df.groupby("_shard", sort=True):
    out_file = OUT_DIR / f"{shard_key}.parquet"
    shard_df.drop(columns=["_shard"]).to_parquet(out_file, index=False)
    output_files.append(out_file)

export_df = export_df.drop(columns=["_shard"])
top_trades_preview   = export_df.head(5).copy()
top_trades_count     = len(export_df)
total_top_train_rows = int(export_df["is_train"].sum())

print(f"Total grouped rows exported:  {top_trades_count:,}")
print(f"  training rows: {total_top_train_rows:,}")
print(f"  test rows:     {top_trades_count - total_top_train_rows:,}")
print(f"Output shards:  {len(output_files):,}")
for f in sorted(output_files):
    print(f"  {f.name}  ({pd.read_parquet(f).shape[0]:,} rows)")
print(f"\nSaved partitioned output → {OUT_DIR}")

Total grouped rows exported:  4,484,112
  training rows: 3,519,974
  test rows:     964,138
Output shards:  16
  0.parquet  (253,331 rows)
  1.parquet  (235,849 rows)
  2.parquet  (295,228 rows)
  3.parquet  (303,786 rows)
  4.parquet  (335,916 rows)
  5.parquet  (232,240 rows)
  6.parquet  (351,602 rows)
  7.parquet  (268,562 rows)
  8.parquet  (287,739 rows)
  9.parquet  (218,537 rows)
  a.parquet  (357,231 rows)
  b.parquet  (272,281 rows)
  c.parquet  (246,544 rows)
  d.parquet  (299,373 rows)
  e.parquet  (231,405 rows)
  f.parquet  (294,488 rows)

Saved partitioned output → ../../data/polygon_trades_processed


In [79]:
pd.set_option("display.max_colwidth", None)
if top_trades_count == 0:
    print("No top-wallet trades found for current date/train split filters.")
    pd.read_parquet(output_files[0]).head(0)
else:
    top_trades_preview

In [80]:
pd.read_parquet('../../data/polygon_trades_processed/0.parquet')

,wallet,condition_id,dt,side,outcome,position,total_quantity,avg_price,trade_value_usdc,final_value_usdc,trade_pnl,copyable_pnl,token_winner,final_price,tx_hash,num_fills,is_train,copyable_qty,avail_copy_total_vol,avail_copy_count
0,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 15:56:11+00:00,BUY,No,1633.00,1633.00,0.87,1418.29,0.00,-1418.29,-0.00,False,0.00,0x69bf60ad4cd2e7b4446c16094c9e73f0e0a00bc002488eae937f07daa247edf8,4,True,0.00,-0.00,0.00
1,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:33:14+00:00,BUY,No,2350.00,717.00,0.87,623.07,0.00,-623.07,-623.07,False,0.00,0xd991f5fd4e7dc6802a905d452d8eb61189d53ee3e7512dc27e4c5fe82065d4a5,1,True,717.00,148.49,2.00
2,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:34:12+00:00,BUY,No,2400.90,50.90,0.87,44.23,0.00,-44.23,-44.23,False,0.00,0x1adf7421a3544fdf4307b1c7766f9556d11ee45c57a316170a11ba6bbad66127,1,True,50.90,148.49,2.00
3,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:34:32+00:00,BUY,No,3493.34,1092.44,0.86,944.96,0.00,-944.96,-0.00,False,0.00,0xa1e278bc950c2f5a4e635358840e42f01f6e06f07fa8641804ab2968a22b5881,1,True,0.00,-0.00,0.00
4,0x000d257d2dc7616feaef4ae0f14600fdf50a758e,0x072acaa7dd352c795948d495e019eec539539d36698225ab98b2f55c692d1b0c,2025-01-11 18:48:32+00:00,BUY,No,4400.35,907.01,0.86,784.56,0.00,-784.56,-0.00,False,0.00,0x9277f707892477c8d41c590ea8ba6868a506e4381f8ebd33a9219e845f53c164,1,True,0.00,-0.00,0.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
253326,0xfffadf38a520cd5a0035ff52d7fceb436a08864b,0x0ed1d3981a2067c9a53364b00d4342e703026926615de49aec9bead2cb2f8393,2025-10-05 13:27:17+00:00,BUY,Yes,703.81,703.81,0.46,323.75,703.81,380.05,27.00,True,1.00,0x26bfc94db0fd5145f711ab7f7e89925cd394dafc7f4bde90a609ff0a6904006f,2,True,50.00,20.00,2.00
253327,0xfffadf38a520cd5a0035ff52d7fceb436a08864b,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-28 06:52:41+00:00,BUY,No,4433.67,4433.67,0.99,4401.61,4433.67,32.06,32.06,True,1.00,0x56b5e1514176e36bca1eb557bac19cc1ba2a67d80d1a293e9cd7477b4d7729a5,2,False,4433.67,419779.69,636.00
253328,0xfffadf38a520cd5a0035ff52d7fceb436a08864b,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-28 06:52:43+00:00,BUY,No,5277.74,844.07,0.99,838.16,844.07,5.91,5.91,True,1.00,0x49d13d8a3002e5eafaa6dbe0dff7a3396a19100d1794bda647d93d041b173a8c,1,False,844.07,223726.95,337.00
253329,0xfffadf38a520cd5a0035ff52d7fceb436a08864b,0x09cbe3e796661a1d820580145488ad2ccb9ad1e720efcd64b448bb77b97007c1,2026-02-28 06:53:59+00:00,BUY,No,9507.82,4230.08,0.99,4187.78,4230.08,42.30,42.30,True,1.00,0x32a4365be9207412dacae9bbf4a2ea61843c9be45e1ee55b80197996ee540bf6,1,False,4230.08,180941.77,94.00


### Unit test: CTF Exchange contract is not treated as a trader

`0x85d355ef32d62b09d2362184f299a7fc662ee1a4` is the Polymarket CTF Exchange
(on-chain matching contract). It can appear in per-side trade rows as a `wallet`
counterparty, but it is **not** a real trader.

This test always enforces that the exchange address is **not** in `top_wallets`.